In [1]:
import time
import numpy as np
import joblib

from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier


In [2]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

# Load Swin-S split (FIXED SPLIT)
X_train = np.load(f"{RESULTS_DIR}/X_train_swinS.npy")
y_train = np.load(f"{RESULTS_DIR}/y_train_swinS.npy")

X_val = np.load(f"{RESULTS_DIR}/X_val_swinS.npy")
y_val = np.load(f"{RESULTS_DIR}/y_val_swinS.npy")

X_test = np.load(f"{RESULTS_DIR}/X_test_swinS.npy")
y_test = np.load(f"{RESULTS_DIR}/y_test_swinS.npy")

print("Data loaded ✅")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Data loaded ✅
Train: (4440, 1408)
Val: (1111, 1408)
Test: (1388, 1408)


In [3]:
xgb_swinS = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


In [4]:
start_train = time.time()

xgb_swinS.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

train_time_swinS = time.time() - start_train
print(f"Training time (Swin-S + XGB): {train_time_swinS:.2f} seconds")


Training time (Swin-S + XGB): 146.99 seconds


In [5]:
start_val = time.time()
_ = xgb_swinS.predict(X_val)
val_time_swinS = time.time() - start_val

print(f"Validation time (Swin-S + XGB): {val_time_swinS:.4f} seconds")


Validation time (Swin-S + XGB): 0.0814 seconds


In [6]:
start_test = time.time()
y_pred = xgb_swinS.predict(X_test)
test_time_swinS = time.time() - start_test

print(f"Testing time (Swin-S + XGB): {test_time_swinS:.4f} seconds")
print("Test Accuracy:", accuracy_score(y_test, y_pred))


Testing time (Swin-S + XGB): 0.0454 seconds
Test Accuracy: 0.94164265129683


In [7]:
np.save(f"{RESULTS_DIR}/runtime_swinS.npy", np.array([
    train_time_swinS,
    val_time_swinS,
    test_time_swinS
]))

print("Runtime results for Swin-S saved ✅")


Runtime results for Swin-S saved ✅


In [8]:
X_test_B = np.load(f"{RESULTS_DIR}/X_test_swinB.npy")
y_test_B = np.load(f"{RESULTS_DIR}/y_test_swinB.npy")

xgb_swinB = joblib.load(f"{MODEL_DIR}/xgb_mobilevit_swinB.pkl")


In [9]:
start_test_B = time.time()
_ = xgb_swinB.predict(X_test_B)
test_time_swinB = time.time() - start_test_B

print(f"Testing time (Swin-B + XGB): {test_time_swinB:.4f} seconds")


Testing time (Swin-B + XGB): 0.0377 seconds


In [10]:
print("\n===== RUNTIME COMPARISON =====")
print(f"Swin-S + XGB | Train: {train_time_swinS:.2f}s | Val: {val_time_swinS:.4f}s | Test: {test_time_swinS:.4f}s")
print(f"Swin-B + XGB | Test only: {test_time_swinB:.4f}s")



===== RUNTIME COMPARISON =====
Swin-S + XGB | Train: 146.99s | Val: 0.0814s | Test: 0.0454s
Swin-B + XGB | Test only: 0.0377s
